# 04 — Visualization Dashboard

Comprehensive visualization of all AdaMed experiment results.

Sections:
1. Source vs Target data distributions
2. Feature space evolution during training
3. Ablation study: alpha parameter sensitivity
4. Summary figure for documentation

In [ ]:
import sys
sys.path.append('..')

import torch
import json
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.manifold import TSNE
from sklearn.decomposition import PCA

from adamed.data.synthetic_generator import ClinicalTimeSeriesGenerator
from adamed.data.preprocessing import simple_dataloader
from adamed.data.heuristics import get_west_african_parameters
from adamed.models.dann import create_dann_for_adamed
from adamed.training.trainer import DANNTrainer
from adamed.evaluation.visualization import COLORS

sns.set_style('whitegrid')
print('Dashboard ready.')

## 1. Data Distribution Overview

In [ ]:
gen = ClinicalTimeSeriesGenerator(n_source=500, n_target_proxy=100, seed=42)
data = gen.generate_experimental_split()
source_data = data['X'][data['source_idx']]
target_data = data['X'][data['target_idx']]

fig, axes = plt.subplots(2, 5, figsize=(22, 8))
t = np.linspace(0, 24, 48)

for f, name in enumerate(data['feature_names']):
    s_mean = np.nanmean(source_data[:, :, f], axis=0)
    s_std = np.nanstd(source_data[:, :, f], axis=0)
    t_mean = np.nanmean(target_data[:, :, f], axis=0)
    t_std = np.nanstd(target_data[:, :, f], axis=0)
    
    axes[0, f].plot(t, s_mean, color=COLORS['source'], linewidth=2, label='Source')
    axes[0, f].fill_between(t, s_mean-s_std, s_mean+s_std, alpha=0.2, color=COLORS['source'])
    axes[0, f].plot(t, t_mean, color=COLORS['target'], linewidth=2, label='Target')
    axes[0, f].fill_between(t, t_mean-t_std, t_mean+t_std, alpha=0.2, color=COLORS['target'])
    axes[0, f].set_title(name, fontsize=11)
    if f == 0:
        axes[0, f].legend(fontsize=8)
    
    s_vals = source_data[:, :, f].flatten()
    t_vals = target_data[:, :, f].flatten()
    axes[1, f].hist(s_vals, bins=40, alpha=0.5, density=True, color=COLORS['source'])
    axes[1, f].hist(t_vals, bins=40, alpha=0.5, density=True, color=COLORS['target'])

plt.suptitle('Data Distribution Dashboard', fontsize=15, y=1.02)
plt.tight_layout()
plt.show()

## 2. Alpha Sensitivity Analysis (Ablation Study)

Testing alpha = 0 (no reversal), 0.5 (moderate), 1.0 (full), 2.0 (aggressive)

In [ ]:
alphas = [0.0, 0.5, 1.0, 2.0]
results = {}

loader = simple_dataloader(data, batch_size=64, normalize=True, flatten=True)

for alpha_val in alphas:
    print(f'Training with alpha={alpha_val}...')
    model = create_dann_for_adamed(time_steps=48, n_features=5)
    trainer = DANNTrainer(model, lr=1e-3, lambda_domain=alpha_val)
    trainer.alpha_schedule = lambda epoch, a=alpha_val: a
    history = trainer.train(loader, n_epochs=30, verbose=False)
    eval_result = trainer.evaluate(loader)
    results[alpha_val] = {
        'history': history,
        'final_label_acc': eval_result['label_acc'],
        'final_domain_acc': eval_result['domain_acc'],
    }
    print(f'  Label acc: {eval_result["label_acc"]:.4f}')

print('Ablation complete.')

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 5))
colors_abl = ['#4CAF50', '#2196F3', '#FF9800', '#F44336']

for alpha_val, color in zip(alphas, colors_abl):
    axes[0].plot(results[alpha_val]['history']['label_acc'],
                 label=f'a={alpha_val}', color=color, linewidth=2)
axes[0].axhline(y=0.5, color='gray', linestyle='--', alpha=0.5)
axes[0].set_xlabel('Epoch'); axes[0].set_ylabel('Label Accuracy')
axes[0].set_title('Task Performance vs Reversal Strength')
axes[0].legend(); axes[0].grid(True, alpha=0.3)

for alpha_val, color in zip(alphas, colors_abl):
    axes[1].plot(results[alpha_val]['history']['label_loss'],
                 label=f'a={alpha_val}', color=color, linewidth=2)
axes[1].set_xlabel('Epoch'); axes[1].set_ylabel('Label Loss')
axes[1].set_title('Task Loss vs Reversal Strength')
axes[1].legend(); axes[1].grid(True, alpha=0.3)

final_accs = [results[a]['final_label_acc'] for a in alphas]
bars = axes[2].bar([f'a={a}' for a in alphas], final_accs, color=colors_abl, alpha=0.8)
axes[2].axhline(y=0.5, color='gray', linestyle='--', alpha=0.5, label='Random')
axes[2].set_ylabel('Final Label Accuracy')
axes[2].set_title('Ablation: Final Accuracy'); axes[2].set_ylim(0, 1.0)
axes[2].legend(); axes[2].grid(True, alpha=0.3, axis='y')

for bar, acc in zip(bars, final_accs):
    axes[2].text(bar.get_x()+bar.get_width()/2., bar.get_height()+0.02,
                 f'{acc:.3f}', ha='center', va='bottom', fontweight='bold')

plt.suptitle('Ablation Study: Gradient Reversal Strength', fontsize=14, y=1.02)
plt.tight_layout()
plt.savefig('../results/figures/ablation_study.png', dpi=150, bbox_inches='tight')
plt.show()

## Summary

Increasing reversal strength monotonically degrades task performance toward
random chance (50%). No alpha value achieves both good task performance AND
domain alignment. This confirms zero-shot DANN is fundamentally limited.

See `docs/next_steps.md` for proposed alternatives.